# CRLocNet benchmark (Google Colab)

Chạy lần lượt các cell bên dưới. Notebook dùng đúng bộ dữ liệu benchmark của AMagPoseNet. `raw_epm`/`synthetic_epm` là tùy chọn: chỉ đặt khi có CSV theoretical EPM 64 cột, cùng thứ tự hàng với dữ liệu tương ứng.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import torch
print(f"PyTorch : {torch.__version__}")
print(f"Device     : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")

In [ ]:
!pip install -q huggingface_hub

import os
from google.colab import userdata
from huggingface_hub import login

# Lấy token từ Secrets của Colab
hf_token = userdata.get('tuankun08')

# Tiến hành đăng nhập
login(token=hf_token)
print("Đã kết nối thành công với Hugging Face!")

In [ ]:
from huggingface_hub import snapshot_download, login
from google.colab import userdata

repo_id="tuankun08/Dataset"

# Tải toàn bộ repository/folder về thư mục '/content/Dataset' trên Colab
snapshot_download(
    repo_id=repo_id,
    allow_patterns="Dataset/*",
    repo_type="dataset",
    local_dir="/content/Dataset"
)

In [ ]:
!git clone https://github.com/tuananhh08/Hallsensor_final.git /content/Hallsensor_final
!pip -q install pandas numpy matplotlib
!pip install optuna --quiet
!pip install -q scikit-learn pandas numpy

DATA_DIR  = "/content/Dataset/Dataset"
REPO      = "/content/Hallsensor_final/Benchmark/CRLocNet"
RUNS      = "/content/drive/MyDrive/CRLocNet_runs"
CACHE     = "/content/crlocnet_cache"

# Finetune data
RAW_VOLTAGE           = f"{DATA_DIR}/Grid_add_Random_data.csv"
CLEAN_VOLTAGE         = f"{DATA_DIR}/Grid_add_Random_data_computed.csv"

# Grid data
GRID_VOLTAGE          = f"{DATA_DIR}/Grid_data.csv"
GRID_CLEAN_VOLTAGE    = f"{DATA_DIR}/Grid_data_computed.csv"

# Helix data
HELIX_VOLTAGE         = f"{DATA_DIR}/Helix_data_2.csv"
HELIX_CLEAN_VOLTAGE   = f"{DATA_DIR}/Helix_data_2_computed.csv"

# Random data
RANDOM_VOLTAGE        = f"{DATA_DIR}/Random_data.csv"
RANDOM_CLEAN_VOLTAGE  = f"{DATA_DIR}/Random_data_computed.csv"

# Label
RAW_LABEL             = f"{DATA_DIR}/Grid_add_Random_points_coordinates.csv"
GRID_LABEL            = f"{DATA_DIR}/Grid_points_coordinates.csv"
HELIX_LABEL           = f"{DATA_DIR}/Helix_points_coordinates_2.csv"
RANDOM_LABEL          = f"{DATA_DIR}/Random_points_coordinates.csv"

# Synthetic data ~700k data
SYNTHETIC_VOLTAGE = f"{DATA_DIR}/synthetic_grid_data.csv"
SYNTHETIC_LABEL   = f"{DATA_DIR}/synthetic_grid_coordinates.csv"

# Calibration parameters path
CALIB_PHYSICAL_CSV = f"{DATA_DIR}/Calibration_Parameters_r_per_sensor.csv"
CALIB_ALPHA_CSV    = f"{DATA_DIR}/Calibration_Alpha_r_per_sensor.csv"


In [ ]:
# STAGE 1 PRE-TRAIN MODNET
!python $REPO/run.py \
  --stage calinet \
  --raw-voltage "$RAW_VOLTAGE" \
  --clean-voltage "$CLEAN_VOLTAGE" \
  --raw-label "$RAW_LABEL" \
  --synthetic-voltage "$SYNTHETIC_VOLTAGE" \
  --synthetic-label "$SYNTHETIC_LABEL" \
  --ckpt-dir "$RUNS/calinet" \
  --cache-dir "$CACHE" \
  --batch-size 64 \
  --num-epochs 260 \
  --lr 0.0007975614143764486 \
  --weight-decay 3.281437096954538e-05 \
  --warmup-epochs 5 \
  --eta-min 1e-6 \
  --patience 45 \
  --save-every 10 \
  --grad-clip-norm 1.0 \
  --calib-reduction mean \
  --no-compile

In [ ]:
# STAGE 2 — LocNet: train synthetic data (giống PoseNet pre-training).
# Raw/clean paths vẫn truyền vào để dùng chung cache/scaler; LocNet chỉ lấy synthetic data + synthetic label.
cmd = f'''cd {REPO} && python -m Benchmark.CRLocNet.run --stage locnet \
  --raw-voltage {RAW_VOLTAGE} --clean-voltage {CLEAN_VOLTAGE} --raw-label {RAW_LABEL} \
  --synthetic-voltage {SYNTHETIC_VOLTAGE} --synthetic-label {SYNTHETIC_LABEL} \
  --ckpt-dir {RUNS}/locnet --cache-dir {CACHE} \
  --batch-size 512 --epochs 200 --lr 1e-4 --weight-decay 4.56e-3 \
  --lambda-direction 0.2 --warmup-epochs 5 --eta-min 1e-6 \
  --patience 45 --save-every 10 --no-compile'''
if RAW_EPM: cmd += f' --raw-epm {RAW_EPM}'
if SYNTHETIC_EPM: cmd += f' --synthetic-epm {SYNTHETIC_EPM}'
!$cmd

In [ ]:
# STAGE 3 — fine-tune end-to-end ModNet + LocNet trên raw/clean/label thật.
cmd = f'''cd {REPO} && python -m Benchmark.CRLocNet.run --stage finetune \
  --raw-voltage {RAW_VOLTAGE} --clean-voltage {CLEAN_VOLTAGE} --raw-label {RAW_LABEL} \
  --synthetic-voltage {SYNTHETIC_VOLTAGE} --synthetic-label {SYNTHETIC_LABEL} \
  --modnet-checkpoint {RUNS}/modnet/modnet_pretrained.pt \
  --locnet-checkpoint {RUNS}/locnet/locnet_pretrained.pt \
  --ckpt-dir {RUNS}/finetune --cache-dir {CACHE} \
  --batch-size 32 --epochs 200 --lr-modnet 1e-4 --lr-locnet 1e-4 \
  --weight-decay 4.56e-3 --lambda-direction 0.2 --lambda-physics 1e-4 \
  --lambda-alignment 0.1 --warmup-epochs 5 --eta-min 1e-6 \
  --patience 45 --save-every 10 --no-compile'''
if RAW_EPM: cmd += f' --raw-epm {RAW_EPM}'
if SYNTHETIC_EPM: cmd += f' --synthetic-epm {SYNTHETIC_EPM}'
# Khi có geometry/moment và raw CSV là Tesla, bỏ comment dòng dưới và cập nhật tham số.
# cmd += ' --physics-informed --sensor-pitch-m 0.01 --moment-am2 YOUR_MOMENT_AM2 --measurement-axis 2'
!$cmd

In [ ]:
# Evaluation: output CSV, 3D trajectory, position/orientation error charts và timing.
EVAL_DIR = f'{RUNS}/evaluation'
os.makedirs(EVAL_DIR, exist_ok=True)
cmd = f'''cd {REPO} && python -m Benchmark.CRLocNet.evaluation.evaluate \
  --test-voltage {HELIX_VOLTAGE} --test-label {HELIX_LABEL} \
  --ckpt-dir {RUNS}/finetune --checkpoint full_model_best.pt \
  --scalers {RUNS}/finetune/scalers.pkl --batch-size 64 --device cuda \
  --out {EVAL_DIR}/crlocnet_trajectory.png \
  --out-csv {EVAL_DIR}/crlocnet_predictions.csv'''
if HELIX_EPM: cmd += f' --test-epm {HELIX_EPM}'
!$cmd

from IPython.display import display, Image
display(Image(filename=f'{EVAL_DIR}/crlocnet_trajectory.png'))
display(Image(filename=f'{EVAL_DIR}/crlocnet_position_error.png'))
display(Image(filename=f'{EVAL_DIR}/crlocnet_orientation_error.png'))

In [ ]:
# Tải artifacts về máy local (tùy chọn).
from google.colab import files
files.download(f'{EVAL_DIR}/crlocnet_predictions.csv')
files.download(f'{EVAL_DIR}/crlocnet_trajectory.png')